# SLC46A1 engraftment, step 8: pairs at chicken 407 in the background the construct has

Step 7 scored the admissible construct as a set and found that four of its five
high-risk targets keep a rescue above 1e-3 while chicken 407 does not. The pair
A404D with C410F was chosen in step 4 against wild-type chicken, where it reached
1.109422e-03, and in the construct it falls to 6.771618e-04. That pair was never
re-searched in the background the residue actually has, so the position is untested
rather than closed.

This run searches it. The background is the finished construct with chicken 407 left
carrying its chicken residue and the two positions the step 4 pair used released back
to chicken, so the search can rediscover that pair or replace it. Every pair over the
union of the ten positions most strongly coupled to 407 and the ten nearest it in the
coordinates is scored, 37905 double variants over 105 pairs, with the 285 singles at
those positions scored in the same pass and carrying their step 2 value beside the new
one.

Two checks run before the sweep and stop it if either fails. The background with the
step 4 pair applied has to be byte for byte the admissible construct with 407 left
alone, and scoring it has to reproduce the value step 7 recorded for chicken 407 in
that construct. A wrong background therefore kills the run in two minutes instead of
three hours.

Every result is written to Drive as it is produced and the run resumes from its
checkpoints, so a disconnected runtime costs only the batch in flight.

In [ ]:
import os, sys, json, hashlib, datetime, time

from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/MIT/All Project Files/SLC46A1 engraftment'
OUT     = os.path.join(PROJECT, 'data', 'esm2_step8')

if not os.path.isdir(PROJECT):
    raise SystemExit('FATAL project folder not found: %s' % PROJECT)
os.makedirs(OUT, exist_ok=True)

REQUIRED = ['data/admissible_construct.json',
            'data/coupling_guided_rescue.json',
            'data/structural_candidates.json',
            'data/pharmacophore_position_set.tsv',
            'data/esm2_step2/rescue_scan.tsv']
missing = [f for f in REQUIRED if not os.path.exists(os.path.join(PROJECT, f))]
if missing:
    raise SystemExit('FATAL missing inputs in %s: %s' % (PROJECT, missing))

S7_CANDIDATES = ['data/esm2_step7/step7_summary.json', 'esm2_step7/step7_summary.json']
found = [f for f in S7_CANDIDATES if os.path.exists(os.path.join(PROJECT, f))]
if not found:
    raise SystemExit('FATAL step 7 summary not found in %s under any of %s. This run is '
                     'tied to step 7 by one value and cannot start without it.'
                     % (PROJECT, S7_CANDIDATES))
if len(found) > 1:
    digests = {f: hashlib.sha256(open(os.path.join(PROJECT, f), 'rb').read()).hexdigest()
               for f in found}
    if len(set(digests.values())) > 1:
        raise SystemExit('FATAL two different step 7 summaries are on Drive, %s. Delete '
                         'the stale one so there is one copy to tie this run to.'
                         % digests)
S7_PATH = os.path.join(PROJECT, found[0])

RUNLOG = os.path.join(OUT, 'run.log')


def log(msg):
    line = '%s  %s' % (datetime.datetime.utcnow().isoformat(timespec='seconds'), msg)
    with open(RUNLOG, 'a') as fh:
        fh.write(line + chr(10))
    print(line, flush=True)


log('start, project=%s' % PROJECT)
log('step 7 summary read from %s' % found[0])
log('output folder %s' % OUT)


In [ ]:
import shutil, subprocess

if shutil.which('nvidia-smi') is None:
    raise SystemExit('FATAL this runtime has no GPU. Set Runtime > Change runtime '
                     'type > T4 GPU (or better), then run from the first cell again.')
if subprocess.run(['nvidia-smi'], capture_output=True).returncode != 0:
    raise SystemExit('FATAL nvidia-smi is present but failing, so the runtime has no '
                     'usable GPU. Change the runtime type and run from the first cell.')

subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'torch', 'transformers'],
               check=True)

import torch
from transformers import AutoTokenizer, EsmForMaskedLM

if not torch.cuda.is_available():
    raise SystemExit('FATAL torch cannot see the GPU after install')
log('torch %s, device %s' % (torch.__version__, torch.cuda.get_device_name(0)))


In [ ]:
import urllib.request

SEQ = os.path.join(OUT, 'seq')
os.makedirs(SEQ, exist_ok=True)


def fetch(url, path):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    urllib.request.urlretrieve(url, path)
    if os.path.getsize(path) == 0:
        raise SystemExit('FATAL empty download: %s' % url)
    return path


fetch('https://rest.uniprot.org/uniprotkb/Q96NT5.fasta', os.path.join(SEQ, 'Q96NT5.fasta'))
fetch('https://www.rcsb.org/fasta/entry/7BC6', os.path.join(SEQ, '7BC6.fasta'))

human = ''.join(l.strip() for l in open(os.path.join(SEQ, 'Q96NT5.fasta'))
                if not l.startswith('>'))
ent = {}
for rec in open(os.path.join(SEQ, '7BC6.fasta')).read().split('>'):
    if rec.strip():
        parts = rec.strip().split(chr(10))
        ent[parts[0].split('|')[0]] = ''.join(parts[1:])
chicken = ent['7BC6_1'][:473]
if len(human) != 459 or len(chicken) != 473:
    raise SystemExit('FATAL sequence lengths %d and %d, expected 459 and 473'
                     % (len(human), len(chicken)))
CHICKEN_SHA = hashlib.sha256(chicken.encode()).hexdigest()
EXPECT_SHA = '5d68e7c91dc3af3cd898a3f459617f3b22670f3dbdf9f798e8d740d8669a8aea'
if CHICKEN_SHA != EXPECT_SHA:
    raise SystemExit('FATAL the chicken sequence differs from the one the earlier steps '
                     'scored: %s against %s' % (CHICKEN_SHA, EXPECT_SHA))
log('sequences ok, chicken_sha256 matches the earlier steps')

AA = 'ACDEFGHIKLMNPQRSTVWY'
THRESHOLD = 1e-3
TOL = 5e-2

rows = [l.rstrip(chr(10)).split(chr(9))
        for l in open(os.path.join(PROJECT, 'data', 'pharmacophore_position_set.tsv'))]
pcol = {k: i for i, k in enumerate(rows[0])}
PSET = {}
for r in rows[1:]:
    cp, hp = int(r[pcol['chicken_pos']]), int(r[pcol['human_pos']])
    if chicken[cp - 1] != r[pcol['chicken_aa']] or human[hp - 1] != r[pcol['human_aa']]:
        raise SystemExit('FATAL the position set disagrees with the sequences at chicken '
                         '%d' % cp)
    PSET[cp] = {'chicken_aa': r[pcol['chicken_aa']], 'human_aa': r[pcol['human_aa']],
                'human_pos': hp, 'differs': r[pcol['differs']] == '1'}
PRIM = sorted(p for p, v in PSET.items() if v['differs'])
if len(PRIM) != 20:
    raise SystemExit('FATAL %d primaries, expected 20' % len(PRIM))

adm = json.load(open(os.path.join(PROJECT, 'data', 'admissible_construct.json')))
if sorted(adm['primaries']) != PRIM:
    raise SystemExit('FATAL the admissible construct lists different primaries from the '
                     'position set')
RESC = {int(k): v for k, v in adm['rescuer_positions'].items()}
for q, y in RESC.items():
    if q in PSET:
        raise SystemExit('FATAL rescuer position %d is in the pharmacophore set, so the '
                         'construct is not the admissible one' % q)
    if y == chicken[q - 1]:
        raise SystemExit('FATAL the rescuer at %d is not a substitution' % q)

s7 = json.load(open(S7_PATH))
if s7['chicken_sha256'] != CHICKEN_SHA:
    raise SystemExit('FATAL step 7 scored a different chicken sequence')
S7 = {x['chicken_pos']: x for x in s7['part_a']['profile']}
log('inputs ok: %d primaries, %d rescuer positions %s, step 7 profile over %d primaries'
    % (len(PRIM), len(RESC), sorted(RESC), len(S7)))


In [ ]:
TARGET = 407
if TARGET not in PRIM:
    raise SystemExit('FATAL chicken %d is not a primary substitution' % TARGET)
if str(TARGET) not in adm['chosen']:
    raise SystemExit('FATAL the admissible construct carries no rescue at chicken %d, so '
                     'there is nothing for this run to re-search' % TARGET)
CH = adm['chosen'][str(TARGET)]
R_T = [int(p) for p in CH['positions']]
R_TO = list(CH['to'])
for q in R_T:
    if q not in RESC:
        raise SystemExit('FATAL position %d is named in the rescue at chicken %d but is '
                         'not a rescuer position of the construct' % (q, TARGET))
KEEP = {q: y for q, y in RESC.items() if q not in R_T}
log('the rescue this run re-searches: %s, at positions %s' % (CH['label'], R_T))
log('rescuer positions held fixed  : %s' % sorted(KEEP))


def build(leave_out, rescuers, extra=()):
    s = list(chicken)
    for p in PRIM:
        if p != leave_out:
            s[p - 1] = PSET[p]['human_aa']
    for q, y in rescuers.items():
        s[q - 1] = y
    for q, y in extra:
        s[q - 1] = y
    return ''.join(s)


BG = build(TARGET, KEEP)
FULL = build(TARGET, RESC)
if BG[TARGET - 1] != chicken[TARGET - 1]:
    raise SystemExit('FATAL the background does not leave chicken %d alone' % TARGET)
for q in R_T:
    if BG[q - 1] != chicken[q - 1]:
        raise SystemExit('FATAL position %d was not released back to chicken' % q)
if build(TARGET, KEEP, tuple(zip(R_T, R_TO))) != FULL:
    raise SystemExit('FATAL the background plus the step 4 pair is not the admissible '
                     'construct, so the two are not comparable')
log('background      : %d positions differ from chicken, %d primaries and %d rescuers'
    % (sum(1 for a, b in zip(BG, chicken) if a != b), len(PRIM) - 1, len(KEEP)))
log('check           : the background plus %s rebuilds the admissible construct exactly'
    % CH['label'])

cg = json.load(open(os.path.join(PROJECT, 'data', 'coupling_guided_rescue.json')))
st = json.load(open(os.path.join(PROJECT, 'data', 'structural_candidates.json')))
if str(TARGET) not in cg['targets'] or str(TARGET) not in st['targets']:
    raise SystemExit('FATAL chicken %d is missing from one of the two candidate rankings'
                     % TARGET)
COUP = [int(r['chicken_pos']) for r in cg['targets'][str(TARGET)]['top_coupled']]
NEAR = [int(c['chicken_pos']) for c in st['targets'][str(TARGET)]['candidates']]
CAND = sorted(set(COUP) | set(NEAR))
if TARGET in CAND:
    raise SystemExit('FATAL the target is in its own candidate list')
for q in R_T:
    if q not in CAND:
        raise SystemExit('FATAL position %d carries the rescue being re-searched but is '
                         'not among the candidates, so the sweep could not rediscover it'
                         % q)
log('  coupled top %d : %s' % (len(COUP), COUP))
log('  nearest top %d : %s' % (len(NEAR), NEAR))
log('  union %d       : %s' % (len(CAND), CAND))
log('  of those, %d lie inside the pharmacophore set: %s'
    % (len([p for p in CAND if p in PSET]), [p for p in CAND if p in PSET]))


In [ ]:
MODEL = 'facebook/esm2_t33_650M_UR50D'
BATCH = 48
SCHEMA = 'v8-pairs-at-407-in-construct'

tok = AutoTokenizer.from_pretrained(MODEL)
model = EsmForMaskedLM.from_pretrained(MODEL).eval().cuda()
AAID = {y: tok.convert_tokens_to_ids(y) for y in AA}
if any(v is None or v < 0 for v in AAID.values()):
    raise SystemExit('FATAL the tokenizer has no id for one of the amino acids')
offset = 1
NTOK = len(chicken) + 2

BG_IDS = tok(BG, return_tensors='pt')['input_ids'][0]
if BG_IDS.shape[0] != NTOK:
    raise SystemExit('FATAL the background tokenizes to %d tokens, expected %d'
                     % (BG_IDS.shape[0], NTOK))
READ_IDX = TARGET - 1 + offset
READ_TOK = AAID[PSET[TARGET]['human_aa']]
if int(BG_IDS[READ_IDX]) != AAID[chicken[TARGET - 1]]:
    raise SystemExit('FATAL the read index does not carry the chicken residue')


def score(variants):
    out = []
    for b in range(0, len(variants), BATCH):
        chunk = variants[b:b + BATCH]
        batch = BG_IDS.repeat(len(chunk), 1)
        for k, v in enumerate(chunk):
            for q, y in v:
                batch[k, q - 1 + offset] = AAID[y]
        with torch.no_grad():
            logits = model(batch.cuda()).logits
        sel = torch.softmax(logits[:, READ_IDX, :].float(), dim=-1)
        out.extend(float(sel[k, READ_TOK]) for k in range(len(chunk)))
    return out


BASE, CHECK = score([(), tuple(zip(R_T, R_TO))])
S7_T = S7[TARGET]['p_admissible']
rel = abs(CHECK - S7_T) / max(CHECK, S7_T, 1e-30)
log('baseline of this background      : %.6e' % BASE)
log('%s scored here              : %.6e' % (CH['label'], CHECK))
log('the same variant in step 7       : %.6e, relative difference %.2e' % (S7_T, rel))
if rel > TOL:
    raise SystemExit('FATAL this run does not reproduce step 7 at chicken %d, %.6e here '
                     'against %.6e there. The background is not the construct step 7 '
                     'scored, so nothing below would be comparable.' % (TARGET, CHECK, S7_T))
log('check           : the run reproduces step 7 at chicken %d' % TARGET)
if BASE >= THRESHOLD:
    raise SystemExit('FATAL chicken %d already clears %.0e in this background without any '
                     'rescue, %.6e, which contradicts step 7. Stop and read the profile.'
                     % (TARGET, THRESHOLD, BASE))


In [ ]:
pairs = [(CAND[a], CAND[b]) for a in range(len(CAND)) for b in range(a + 1, len(CAND))]
doubles = [(i, x, j, y) for (i, j) in pairs
           for x in AA if x != BG[i - 1]
           for y in AA if y != BG[j - 1]]
singles = [(j, y) for j in CAND for y in AA if y != BG[j - 1]]
log('pairs %d, doubles %d, singles %d' % (len(pairs), len(doubles), len(singles)))


def sweep(variants, ckpt, header, key_of, as_subs):
    done = set()
    if os.path.exists(ckpt) and os.path.getsize(ckpt) > 0:
        if open(ckpt).readline().rstrip(chr(10)) != header:
            raise SystemExit('FATAL checkpoint %s was written by a different schema; '
                             'delete it and rerun' % ckpt)
        for line in open(ckpt):
            p = line.rstrip(chr(10)).split(chr(9))
            if p[0] == 'row':
                done.add(key_of(p))
        log('resuming %s, already scored %d' % (os.path.basename(ckpt), len(done)))
    todo = [v for v in variants if key_of(['row'] + [str(x) for x in v]) not in done]
    log('%s: %d to score of %d' % (os.path.basename(ckpt), len(todo), len(variants)))
    t0 = time.time()
    with open(ckpt, 'a') as fh:
        if not done:
            fh.write(header + chr(10))
            fh.flush()
        for b in range(0, len(todo), BATCH):
            chunk = todo[b:b + BATCH]
            vals = score([as_subs(v) for v in chunk])
            for v, p in zip(chunk, vals):
                fh.write(chr(9).join(['row'] + [str(x) for x in v] + ['%.6e' % p])
                         + chr(10))
            fh.flush()
            if (b // BATCH) % 50 == 0:
                el = (time.time() - t0) / 60
                rate = (b + len(chunk)) / max(el, 1e-9)
                log('  %d of %d, %.1f min elapsed, about %.0f min left'
                    % (b + len(chunk), len(todo), el,
                       (len(todo) - b - len(chunk)) / max(rate, 1e-9)))
    log('%s complete, %.1f min' % (os.path.basename(ckpt), (time.time() - t0) / 60))


SP = os.path.join(OUT, 'c407_singles_partial.tsv')
DP = os.path.join(OUT, 'c407_doubles_partial.tsv')
HS = chr(9).join(['schema_' + SCHEMA, 'pos', 'aa', 'p_target'])
HD = chr(9).join(['schema_' + SCHEMA, 'pos_i', 'aa_i', 'pos_j', 'aa_j', 'p_target'])
sweep(singles, SP, HS, lambda p: (int(p[1]), p[2]),
      lambda v: ((v[0], v[1]),))
sweep(doubles, DP, HD, lambda p: (int(p[1]), p[2], int(p[3]), p[4]),
      lambda v: ((v[0], v[1]), (v[2], v[3])))


In [ ]:
sing = {}
for line in open(SP):
    p = line.rstrip(chr(10)).split(chr(9))
    if p[0] == 'row':
        sing[(int(p[1]), p[2])] = float(p[3])
dbl = {}
for line in open(DP):
    p = line.rstrip(chr(10)).split(chr(9))
    if p[0] == 'row':
        dbl[(int(p[1]), p[2], int(p[3]), p[4])] = float(p[5])
if len(sing) != len(singles) or len(dbl) != len(doubles):
    raise SystemExit('FATAL the checkpoints hold %d singles and %d doubles, %d and %d '
                     'expected' % (len(sing), len(dbl), len(singles), len(doubles)))

KEY4 = (R_T[0], R_TO[0], R_T[1], R_TO[1])
if KEY4 not in dbl:
    raise SystemExit('FATAL the sweep does not contain the step 4 pair %s' % CH['label'])
rel = abs(dbl[KEY4] - CHECK) / max(dbl[KEY4], CHECK, 1e-30)
if rel > TOL:
    raise SystemExit('FATAL the sweep scores %s at %.6e where the check before it gave '
                     '%.6e' % (CH['label'], dbl[KEY4], CHECK))
log('check           : the sweep reproduces %s at %.6e' % (CH['label'], dbl[KEY4]))

PREV = {}
scan = open(os.path.join(PROJECT, 'data', 'esm2_step2', 'rescue_scan.tsv'))
head = scan.readline().rstrip(chr(10)).split(chr(9))
scol = {k: i for i, k in enumerate(head)}
for line in scan:
    r = line.rstrip(chr(10)).split(chr(9))
    if int(r[scol['target_pos']]) == TARGET:
        PREV[(int(r[scol['rescuer_pos']]), r[scol['rescuer_to']])] = \
            float(r[scol['p_target']])

cols = ['pos_i', 'from_construct_i', 'from_chicken_i', 'to_i',
        'pos_j', 'from_construct_j', 'from_chicken_j', 'to_j',
        'p_target', 'p_single_i', 'p_single_j', 'best_single_of_pair',
        'fold_vs_baseline', 'fold_vs_best_single_of_pair', 'fold_vs_step4_pair',
        'crosses', 'in_set_i', 'in_set_j', 'admissible']
out = []
for (i, x, j, y), p in dbl.items():
    pi, pj = sing[(i, x)], sing[(j, y)]
    out.append({'pos_i': i, 'from_construct_i': BG[i - 1], 'from_chicken_i': chicken[i - 1],
                'to_i': x,
                'pos_j': j, 'from_construct_j': BG[j - 1], 'from_chicken_j': chicken[j - 1],
                'to_j': y,
                'p_target': p, 'p_single_i': pi, 'p_single_j': pj,
                'best_single_of_pair': max(pi, pj),
                'fold_vs_baseline': p / BASE,
                'fold_vs_best_single_of_pair': p / max(pi, pj),
                'fold_vs_step4_pair': p / dbl[KEY4],
                'crosses': p >= THRESHOLD,
                'in_set_i': i in PSET, 'in_set_j': j in PSET,
                'admissible': (i not in PSET) and (j not in PSET)})
out.sort(key=lambda r: -r['p_target'])
with open(os.path.join(OUT, 'c407_double_rescue.tsv'), 'w') as fh:
    fh.write(chr(9).join(cols) + chr(10))
    for r in out:
        fh.write(chr(9).join(('%.6e' % r[c]) if isinstance(r[c], float) else str(r[c])
                             for c in cols) + chr(10))
with open(os.path.join(OUT, 'c407_singles.tsv'), 'w') as fh:
    fh.write(chr(9).join(['pos', 'from_construct', 'from_chicken', 'to', 'p_construct',
                          'p_wt_step2', 'in_set']) + chr(10))
    for (j, y), v in sorted(sing.items()):
        pv = PREV.get((j, y))
        fh.write(chr(9).join([str(j), BG[j - 1], chicken[j - 1], y, '%.6e' % v,
                              ('%.6e' % pv) if pv else '', str(j in PSET)]) + chr(10))

cross = [r for r in out if r['crosses']]
adm_cross = [r for r in cross if r['admissible']]
better = [r for r in out if r['p_target'] > dbl[KEY4]]
adm_better = [r for r in better if r['admissible']]
best_single = max(sing.items(), key=lambda kv: kv[1])
folds = sorted(r['fold_vs_best_single_of_pair'] for r in out)
log('')
log('chicken %d by pairs in the background the construct has' % TARGET)
log('  baseline        : %.6e, %s at %.6e' % (BASE, CH['label'], dbl[KEY4]))
log('  singles         : %d, strongest %s%d%s at %.6e, inside the set %s'
    % (len(sing), BG[best_single[0][0] - 1], best_single[0][0], best_single[0][1],
       best_single[1], best_single[0][0] in PSET))
log('  doubles         : %d over %d pairs, %d crossing %.0e, %d of those sparing the set'
    % (len(out), len(pairs), len(cross), THRESHOLD, len(adm_cross)))
log('  beating %s: %d, of which %d spare the set'
    % (CH['label'], len(better), len(adm_better)))
log('  fold over the better half: median %.2f, max %.2f'
    % (folds[len(folds) // 2], max(folds)))
log('  the five strongest')
for r in out[:5]:
    log('    %s%d%s with %s%d%s at %.6e, %.1f-fold over baseline, %.2f-fold over the '
        'better half, spares the set %s'
        % (r['from_construct_i'], r['pos_i'], r['to_i'],
           r['from_construct_j'], r['pos_j'], r['to_j'], r['p_target'],
           r['fold_vs_baseline'], r['fold_vs_best_single_of_pair'], r['admissible']))
if adm_cross:
    r = adm_cross[0]
    log('  strongest crossing pair that spares the set: %s%d%s with %s%d%s at %.6e'
        % (r['from_construct_i'], r['pos_i'], r['to_i'],
           r['from_construct_j'], r['pos_j'], r['to_j'], r['p_target']))
else:
    log('  no crossing pair leaves the pharmacophore set untouched')
log('  verdict         : %s'
    % ('chicken %d is rescued in the construct background by a pair that spares the set'
       % TARGET if adm_cross else
       ('chicken %d is rescued only by a pair that overwrites the set' % TARGET
        if cross else
        'no pair at these %d positions lifts chicken %d above %.0e in the background the '
        'construct has' % (len(CAND), TARGET, THRESHOLD))))

summary = {'schema': SCHEMA, 'model': MODEL, 'chicken_sha256': CHICKEN_SHA,
           'threshold': THRESHOLD, 'target': TARGET,
           'human_pos': PSET[TARGET]['human_pos'], 'human_aa': PSET[TARGET]['human_aa'],
           'background': {'primaries_in': len(PRIM) - 1,
                          'rescuer_positions_held': {str(k): v for k, v in KEEP.items()},
                          'positions_released': R_T,
                          'differs_from_chicken': sum(1 for a, b in zip(BG, chicken)
                                                      if a != b),
                          'sequence': BG},
           'checks': {'step7_value': S7_T, 'step4_pair_here': dbl[KEY4],
                      'baseline': BASE,
                      'rebuilds_admissible_construct': True},
           'step4_pair': {'label': CH['label'], 'positions': R_T, 'to': R_TO,
                          'p_wt_background': CH['p'], 'p_construct': dbl[KEY4]},
           'coupled': COUP, 'nearest': NEAR, 'candidates': CAND,
           'candidates_in_set': [p for p in CAND if p in PSET],
           'n_pairs': len(pairs), 'n_doubles': len(out), 'n_singles': len(sing),
           'best_single': {'pos': best_single[0][0], 'to': best_single[0][1],
                           'p': best_single[1],
                           'in_set': best_single[0][0] in PSET},
           'n_crossing': len(cross), 'n_crossing_sparing_the_set': len(adm_cross),
           'n_better_than_step4_pair': len(better),
           'n_better_sparing_the_set': len(adm_better),
           'best_double': out[0],
           'best_sparing_double': adm_cross[0] if adm_cross else None,
           'top_doubles': out[:20],
           'fold_over_better_half': {'median': folds[len(folds) // 2], 'max': max(folds)}}
with open(os.path.join(OUT, 'step8_summary.json'), 'w') as fh:
    json.dump(summary, fh, indent=1)
log('')
log('wrote c407_double_rescue.tsv, c407_singles.tsv and step8_summary.json')
log('copy data/esm2_step8/ from Drive into the local data/ and run '
    'codes/22_step8_read.py')
